# ⚖️ DLKR-LJP V5.0: Dynamic Legal Knowledge Retrieval for Legal Judgment Prediction
### Pretrained Open Legal Weights: InLegalBERT + OpenNyAI Transformer NER + Qwen2.5-7B (4-Bit NF4 GPU)
**Jurisprudence:** Indian Penal Code (IPC 1860) & Bharatiya Nyaya Sanhita (BNS 2023)  
**Hardware Target:** Kaggle GPU (T4 x 2 or P100) / Google Colab (T4 / A100)  
**Features:** Dual-Engine Reasoning, Statutory Bound Verification (JCV), Prefix Knowledge Fusion, Gradio 6.x UI

---
> ⚠️ **Research & Educational Prototype:** Not Legal Advice. All predictions and judicial ratios are computed for research purposes.

## 1. Hardware Inspection & Dependency Installation
Installs PyTorch, Hugging Face Transformers, BitsAndBytes 4-bit quantization, Sentence-Transformers, Gradio 6.x, and the official Supreme Court OpenNyAI Legal NER transformer model.

In [ ]:
!nvidia-smi
!pip install -q torch transformers bitsandbytes accelerate sentence-transformers gradio spacy
# Install official OpenNyAI Legal NER model for Indian jurisprudence
!pip install -q https://huggingface.co/opennyaiorg/en_legal_ner_trf/resolve/main/en_legal_ner_trf-any-py3-none-any.whl
print('All dependencies successfully installed!')

## 2. Deploy DLKR-LJP Self-Contained Pipeline Engine
Writes and initializes the standalone DLKR-LJP v5.0 pipeline script (`standalone_dlkr_ljp.py`) into the runtime environment.

In [ ]:
import base64

b64_data = """..."""
script_content = base64.b64decode(b64_data).decode('utf-8')

with open('standalone_dlkr_ljp.py', 'w', encoding='utf-8') as f:
    f.write(script_content)

print(f'standalone_dlkr_ljp.py deployed successfully ({len(script_content):,} chars).')

## 3. Run Automated 11-Case Benchmark Verification Suite
Executes all 11 benchmark legal fact narratives spanning theft, sudden fight homicide, cheating & financial fraud, knife robbery, matrimonial cruelty, civil quashing, gunpoint robbery, knife stabbing, job scams, joint attacks, and non-legal abstention.

In [ ]:
import sys
import importlib.util

spec = importlib.util.spec_from_file_location('dlkr_ljp', 'standalone_dlkr_ljp.py')
dlkr = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dlkr)

# Initialize pipeline with GPU acceleration
pipeline = dlkr.get_pipeline(load_transformer=True)

# Execute full benchmark suite
success = dlkr.run_benchmark_suite(pipeline)
print(f'\nBenchmark Suite All Passed: {success}')

## 4. Launch Interactive Gradio 6.x Web Application
Launches the interactive legal adjudication interface with a public Gradio shareable link (`https://...gradio.live`) for browser testing.

In [ ]:
import os

demo = dlkr.build_gradio_dashboard()
if demo is not None:
    demo.queue(default_concurrency_limit=5).launch(
        server_name='0.0.0.0',
        server_port=7860,
        share=True,
        show_error=True
    )
else:
    print('Gradio is not installed.')